In [ ]:
import numpy as np
import scipy.stats as si

def black_scholes_call(S, K, T, r, sigma):
    
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2)
    return call_price

from scipy.stats import norm
from scipy.optimize import brentq

def implied_volatility(S, K, T, r, C):
    #Calculate the implied volatility of an option using the Black-Scholes formula.
    def f(sigma):
        d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2) - C
    return brentq(f, 1e-6, 1)

S = 100.0  
K = 100.0  
T = 0.5  
r = 0.01  
C = 10.0  

implied_vol = implied_volatility(S, K, T, r, C)
print(f"Implied volatility: {implied_vol:.2%}") 

In [ ]:
import yfinance as yf
import pandas as pd

with open('/Users/Farid Srouji/Documents/Research/master/Data/CSV/companylist.csv') as f:
    companies = f.read().splitlines()

rows=[]
for company in companies:
    stock_data = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/Data/PKL/Stocks/"+company+".pkl")
    stock_data.sort_values(by="date",inplace=True)
    stock_data["date"] = pd.to_datetime(stock_data["date"])
    dates = [date_obj.strftime('%Y-%m-%d') for date_obj  in stock_data.date]
    for date in dates:
        print(date)
        try:
            ticker = yf.Ticker(company)
            option_chain = ticker.option_chain(date)
            S = stock_data["adjClose"] 
            K = stock_data["adjClose"] 
            T = 0.5  
            r = 0.01  
            C = 10.0  
            implied_vol = implied_volatility(S, K, T, r, C)
            row=[company, implied_vol]   
            rows.append(row)
            print(f"Implied volatility: {implied_vol:.2%}")
        except:
            print("Exception"+date)


In [ ]:
df = pd.DataFrame(data=rows)